# Spotify Global Weekly Chart Scraper

Scrapes Spotify's public weekly chart data (Global Top 200) from [charts.spotify.com](https://charts.spotify.com) using Selenium, then saves the results as monthly CSV files.

## Workflow
1. **Save login cookies** – Open a browser, log in manually, then persist session cookies to disk.
2. **Scrape weekly charts** – For every Thursday from the start date to today, navigate to the chart page, click the CSV download button, read the file, and tag it with the chart week.
3. **Batch by month** – Aggregate each month's weekly DataFrames and write one CSV per month (e.g. `spotify_weekly_data_2021-01.csv`).

## Requirements
```
pip install selenium webdriver-manager pandas
```

## 1. Imports & Configuration

In [1]:
import glob
import os
import pickle
import time
from collections import defaultdict
from datetime import datetime, timedelta

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ── Configuration ─────────────────────────────────────────────────────────────
SCRIPT_DIR   = os.getcwd()                                   # output directory
COOKIE_FILE  = os.path.join(SCRIPT_DIR, "spotify_cookies.pkl")
START_DATE   = "2021-01-01"                                  # ← change as needed
DOWNLOAD_DIR = os.path.join(SCRIPT_DIR, "downloads_temp")

print(f"Working directory : {SCRIPT_DIR}")
print(f"Cookie file       : {COOKIE_FILE}")
print(f"Start date        : {START_DATE}")

Working directory : /Users/tinalin/Downloads
Cookie file       : /Users/tinalin/Downloads/spotify_cookies.pkl
Start date        : 2021-01-01


## 2. Helper Utilities

In [3]:
def create_driver(download_dir=None):
    """Create a Chrome WebDriver instance with an optional download directory."""
    options = Options()
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    if download_dir:
        os.makedirs(download_dir, exist_ok=True)
        options.add_experimental_option("prefs", {
            "download.default_directory": download_dir,
            "download.prompt_for_download": False,
            "download.directory_upgrade": True,
        })
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )


def wait_for_download(download_dir, timeout=10):
    """Block until a new .csv appears in download_dir (ignores partial files)."""
    existing = set(glob.glob(os.path.join(download_dir, "*.csv")))
    elapsed  = 0
    while elapsed < timeout:
        time.sleep(0.5)
        elapsed += 0.5
        current   = set(glob.glob(os.path.join(download_dir, "*.csv")))
        new_files = current - existing
        if new_files:
            path = max(new_files, key=os.path.getmtime)
            if not path.endswith(".crdownload"):
                return path
    return None


def load_cookies(driver):
    """Inject saved cookies into the current Selenium session."""
    if not os.path.exists(COOKIE_FILE):
        raise FileNotFoundError(
            f"Cookie file not found at {COOKIE_FILE}.\n"
            "Run the 'Save Cookies' cell first."
        )
    driver.get("https://accounts.spotify.com/en/login")
    time.sleep(2)
    with open(COOKIE_FILE, "rb") as f:
        cookies = pickle.load(f)
    for cookie in cookies:
        cookie.pop("domain", None)
        try:
            driver.add_cookie(cookie)
        except Exception:
            continue
    driver.refresh()
    time.sleep(5)
    print("✅ Cookies loaded — session restored.")


def generate_weekly_thursdays(start_date, end_date=None):
    """Yield every Thursday from start_date up to end_date (default: today)."""
    start            = datetime.strptime(start_date, "%Y-%m-%d")
    end              = end_date or datetime.now()
    days_to_thursday = (3 - start.weekday()) % 7
    current          = start + timedelta(days=days_to_thursday)
    while current <= end:
        yield current
        current += timedelta(days=7)


print("Helper functions defined.")

Helper functions defined.


## 3. Step 1 — Save Login Cookies

> **Run this cell only once** (or whenever your session expires).  
> A Chrome window will open — log in to Spotify, wait for the page to fully load, then come back here and press **Enter**.

In [4]:
def save_cookies():
    """Open a browser for manual login, then persist session cookies."""
    driver = create_driver()
    driver.get("https://accounts.spotify.com/en/login")
    input(
        "\n>>> Log in to Spotify in the browser window, wait for the page "
        "to fully load, then press ENTER here to save cookies... "
    )
    with open(COOKIE_FILE, "wb") as f:
        pickle.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {COOKIE_FILE}")
    driver.quit()


save_cookies()


>>> Log in to Spotify in the browser window, wait for the page to fully load, then press ENTER here to save cookies...  


✅ Cookies saved to /Users/tinalin/Downloads/spotify_cookies.pkl


## 4. Step 2 — Scrape Weekly Charts

Downloads every weekly chart CSV from `START_DATE` to today and saves one CSV per calendar month.  
Make sure you have run the **Save Cookies** cell (or have a valid `spotify_cookies.pkl`) before running this.

In [6]:
def scrape_charts(start_date=START_DATE, download_dir=DOWNLOAD_DIR):
    """Download every weekly chart CSV from start_date to today."""
    driver = create_driver(download_dir=download_dir)

    try:
        load_cookies(driver)
    except FileNotFoundError as e:
        print(e)
        driver.quit()
        return

    # Group Thursdays by year-month → one output CSV per month
    months = defaultdict(list)
    for dt in generate_weekly_thursdays(start_date):
        months[dt.strftime("%Y-%m")].append(dt.strftime("%Y-%m-%d"))

    total_weeks = sum(len(v) for v in months.values())
    print(f"Scraping {total_weeks} weeks across {len(months)} months ...\n")

    for year_month, weeks in months.items():
        month_frames = []
        print(f"--- {year_month} ({len(weeks)} weeks) ---")

        for date_str in weeks:
            url = f"https://charts.spotify.com/charts/view/regional-global-weekly/{date_str}"
            driver.get(url)

            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.TAG_NAME, "table"))
                )
                time.sleep(1)

                # Locate the CSV download button
                download_btn = None
                for btn in driver.find_elements(
                    By.CSS_SELECTOR, 'button[data-encore-id="buttonTertiary"]'
                ):
                    if btn.find_elements(By.TAG_NAME, "svg") and btn.is_displayed():
                        download_btn = btn
                        break

                if download_btn:
                    download_btn.click()
                    csv_path = wait_for_download(download_dir)
                    if csv_path:
                        df = pd.read_csv(csv_path)
                        df["Week_Ending"] = date_str
                        month_frames.append(df)
                        os.remove(csv_path)
                        print(f"  [OK]   {date_str}  ({len(df)} rows)")
                    else:
                        print(f"  [FAIL] {date_str}  (download timed out)")
                else:
                    print(f"  [FAIL] {date_str}  (download button not found)")

                time.sleep(1)

            except Exception as e:
                print(f"  [FAIL] {date_str}  ({e})")

        # Save monthly CSV
        if month_frames:
            combined = pd.concat(month_frames, ignore_index=True)
            out_path = os.path.join(SCRIPT_DIR, f"spotify_weekly_data_{year_month}.csv")
            combined.to_csv(out_path, index=False)
            print(f"  => Saved {len(combined)} rows to {out_path}")
        else:
            print(f"  => WARNING: no data collected for {year_month}")

        time.sleep(5)  # polite delay between months

    driver.quit()

    # Clean up temp folder if empty
    if os.path.isdir(download_dir) and not os.listdir(download_dir):
        os.rmdir(download_dir)

    print("\n✅ All done!")


scrape_charts()

✅ Cookies loaded — session restored.
Scraping 269 weeks across 62 months ...

--- 2021-01 (4 weeks) ---
  [FAIL] 2021-01-07  (download timed out)
  [FAIL] 2021-01-14  (download timed out)
  [FAIL] 2021-01-21  (download timed out)
  [FAIL] 2021-01-28  (download timed out)
  => WARNING: no data collected for 2021-01
--- 2021-02 (4 weeks) ---
  [FAIL] 2021-02-04  (Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=145.0.7632.117)
Stacktrace:
0   chromedriver                        0x00000001006456b4 cxxbridge1$str$ptr + 3127600
1   chromedriver                        0x000000010063da50 cxxbridge1$str$ptr + 3095756
2   chromedriver                        0x000000010011a56c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75432
3   chromedriver                        0x00000001000f2eb4 chromedriver + 159412
4   chromedriver                        0x000000010018c274 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_al

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=145.0.7632.117)
Stacktrace:
0   chromedriver                        0x00000001006456b4 cxxbridge1$str$ptr + 3127600
1   chromedriver                        0x000000010063da50 cxxbridge1$str$ptr + 3095756
2   chromedriver                        0x000000010011a56c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75432
3   chromedriver                        0x00000001000f2eb4 chromedriver + 159412
4   chromedriver                        0x000000010018c274 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 541616
5   chromedriver                        0x00000001001a2148 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 631428
6   chromedriver                        0x0000000100157b9c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 326872
7   chromedriver                        

## 5. (Optional) Preview a Monthly CSV

In [ ]:
# Change the filename below to whichever month you'd like to inspect
preview_file = os.path.join(SCRIPT_DIR, "spotify_weekly_data_2021-01.csv")

if os.path.exists(preview_file):
    df_preview = pd.read_csv(preview_file)
    print(f"Shape: {df_preview.shape}")
    display(df_preview.head(10))
else:
    print(f"File not found: {preview_file}")